# 1.Importing Libraries

In [1]:
%pip install numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install numpy numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split

# 2. Reading the Data

In [2]:
PROJECT_DIR = r"C:\flight_price prediction_AWS_sagemaker"
DATA_DIR = "Dataset"

In [3]:
def get_data(name):
	file_name = f"{name}.csv"
	file_path = os.path.join(PROJECT_DIR, DATA_DIR, file_name)
	return pd.read_csv(file_path)

In [4]:
flights = get_data("flight_price")
flights

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,3/22/2026 1:10,2h 50m,non-stop,No info,3897
1,Air India,1/5/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,5:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/6/2019,Delhi,Cochin,DEL → LKO → BOM → COK,9:25,6/10/2026 4:25,19h,2 stops,No info,13882
3,IndiGo,12/5/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,1/3/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302
...,...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,9/4/2019,Kolkata,Banglore,CCU → BLR,19:55,22:25,2h 30m,non-stop,No info,4107
10679,Air India,27/04/2019,Kolkata,Banglore,CCU → BLR,20:45,23:20,2h 35m,non-stop,No info,4145
10680,Jet Airways,27/04/2019,Banglore,Delhi,BLR → DEL,8:20,11:20,3h,non-stop,No info,7229
10681,Vistara,1/3/2019,Banglore,New Delhi,BLR → DEL,11:30,14:10,2h 40m,non-stop,No info,12648


In [5]:
flights.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Date_of_Journey  10683 non-null  str  
 2   Source           10683 non-null  str  
 3   Destination      10683 non-null  str  
 4   Route            10682 non-null  str  
 5   Dep_Time         10683 non-null  str  
 6   Arrival_Time     10683 non-null  str  
 7   Duration         10683 non-null  str  
 8   Total_Stops      10682 non-null  str  
 9   Additional_Info  10683 non-null  str  
 10  Price            10683 non-null  int64
dtypes: int64(1), str(10)
memory usage: 1.8 MB


# 3. Preliminary Analysis

In [6]:
flights.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,3/22/2026 1:10,2h 50m,non-stop,No info,3897
1,Air India,1/5/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,5:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/6/2019,Delhi,Cochin,DEL → LKO → BOM → COK,9:25,6/10/2026 4:25,19h,2 stops,No info,13882
3,IndiGo,12/5/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,1/3/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [7]:
flights.dtypes

Airline              str
Date_of_Journey      str
Source               str
Destination          str
Route                str
Dep_Time             str
Arrival_Time         str
Duration             str
Total_Stops          str
Additional_Info      str
Price              int64
dtype: object

- The dataset contains 10,683 rows and 11 features
- Columns Route and Total_Stops have missing value each.
- The data types of some features isn't appropriate

In [8]:
value = flights.Date_of_Journey.iloc[6]
value

'12/3/2019'

In [9]:
value = flights.Dep_Time.iloc[6]
value

'18:55'

3.2 Check for Duplicates

In [10]:
print(flights.duplicated().sum())

220


In [11]:
(
	flights
	.loc[flights.duplicated(keep=False)]
	.sort_values(["Airline", "Date_of_Journey", "Source", "Destination"])
)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
6321,Air India,1/3/2019,Banglore,New Delhi,BLR → BOM → AMD → DEL,8:50,3/2/2026 23:55,39h 5m,2 stops,No info,17135
9848,Air India,1/3/2019,Banglore,New Delhi,BLR → BOM → AMD → DEL,8:50,3/2/2026 23:55,39h 5m,2 stops,No info,17135
1495,Air India,1/4/2019,Kolkata,Banglore,CCU → DEL → COK → BLR,10:00,4/2/2026 1:20,15h 20m,2 stops,No info,10408
9913,Air India,1/4/2019,Kolkata,Banglore,CCU → DEL → COK → BLR,10:00,4/2/2026 1:20,15h 20m,2 stops,No info,10408
3598,Air India,1/5/2019,Kolkata,Banglore,CCU → GAU → DEL → BLR,9:50,5/2/2026 8:55,23h 5m,2 stops,No info,13227
...,...,...,...,...,...,...,...,...,...,...,...
3711,SpiceJet,24/03/2019,Banglore,New Delhi,BLR → DEL,20:30,23:20,2h 50m,non-stop,No check-in baggage included,3873
7395,SpiceJet,3/3/2019,Banglore,New Delhi,BLR → DEL,20:30,23:20,2h 50m,non-stop,No info,6860
7502,SpiceJet,3/3/2019,Banglore,New Delhi,BLR → DEL,20:30,23:20,2h 50m,non-stop,No info,6860
2634,Vistara,24/03/2019,Banglore,New Delhi,BLR → DEL,11:30,14:10,2h 40m,non-stop,No info,5403


    3.3 Observation

        - The type of Date_of_Journey, Dep_Time and Arrival_Time should be changed to datetime
        - The type of Duration and Total_Stops is mixed. It should be numeric type.
        - There're 220 duplicates. These should be removed.

# 4. Detailed Analysis

       Airline

In [12]:
flights.Airline

0             IndiGo
1          Air India
2        Jet Airways
3             IndiGo
4             IndiGo
            ...     
10678       Air Asia
10679      Air India
10680    Jet Airways
10681        Vistara
10682      Air India
Name: Airline, Length: 10683, dtype: str

In [13]:
flights.Airline.unique()

<ArrowStringArray>
[                           'IndiGo',                         'Air India',
                       'Jet Airways',                          'SpiceJet',
                 'Multiple carriers',                             'GoAir',
                           'Vistara',                          'Air Asia',
           'Vistara Premium economy',              'Jet Airways Business',
 'Multiple carriers Premium economy',                            'Trujet']
Length: 12, dtype: str

In [14]:
(
	flights
	.Airline
	.str.replace(" Premium economy", "")
	.str.replace(" Business", "")
	.str.title()
)

0             Indigo
1          Air India
2        Jet Airways
3             Indigo
4             Indigo
            ...     
10678       Air Asia
10679      Air India
10680    Jet Airways
10681        Vistara
10682      Air India
Name: Airline, Length: 10683, dtype: str

  - Some of the entries have inconsistent/inaccurate values

 Date_of_Journey

In [15]:
flights.Date_of_Journey

0        24/03/2019
1          1/5/2019
2          9/6/2019
3         12/5/2019
4          1/3/2019
            ...    
10678      9/4/2019
10679    27/04/2019
10680    27/04/2019
10681      1/3/2019
10682      9/5/2019
Name: Date_of_Journey, Length: 10683, dtype: str

In [16]:
pd.to_datetime(flights.Date_of_Journey, dayfirst=True)

0       2019-03-24
1       2019-05-01
2       2019-06-09
3       2019-05-12
4       2019-03-01
           ...    
10678   2019-04-09
10679   2019-04-27
10680   2019-04-27
10681   2019-03-01
10682   2019-05-09
Name: Date_of_Journey, Length: 10683, dtype: datetime64[us]

Source

In [17]:
flights.Source.unique()

<ArrowStringArray>
['Banglore', 'Kolkata', 'Delhi', 'Chennai', 'Mumbai']
Length: 5, dtype: str

Destination

In [18]:
flights.Destination.unique()

<ArrowStringArray>
['New Delhi', 'Banglore', 'Cochin', 'Kolkata', 'Delhi', 'Hyderabad']
Length: 6, dtype: str

Dep_time

In [19]:
flights.Dep_Time

0        22:20
1         5:50
2         9:25
3        18:05
4        16:50
         ...  
10678    19:55
10679    20:45
10680     8:20
10681    11:30
10682    10:55
Name: Dep_Time, Length: 10683, dtype: str

In [20]:
(
	flights
	.Dep_Time
	.loc[lambda ser: ser.str.contains("[^0-9:]")]
)

Series([], Name: Dep_Time, dtype: str)

In [21]:
pd.to_datetime(flights.Dep_Time,format="mixed",
    errors="coerce").dt.time

0        22:20:00
1        05:50:00
2        09:25:00
3        18:05:00
4        16:50:00
           ...   
10678    19:55:00
10679    20:45:00
10680    08:20:00
10681    11:30:00
10682    10:55:00
Name: Dep_Time, Length: 10683, dtype: object

Arrival_Time

In [22]:
flights.Arrival_Time

0        3/22/2026 1:10
1                 13:15
2        6/10/2026 4:25
3                 23:30
4                 21:35
              ...      
10678             22:25
10679             23:20
10680             11:20
10681             14:10
10682             19:15
Name: Arrival_Time, Length: 10683, dtype: str

In [23]:
(
	flights
	.Arrival_Time
	.loc[lambda ser: ser.str.contains("[^0-9:]")]
	.str.split(" ", n=1)
	.str.get(1)
	.unique()
)

array(['1:10', '4:25', '10:25', '5:05', '9:20', '19:15', '12:35', '13:20',
       '8:15', '3:35', '19:00', '11:05', '10:55', '0:15', '2:45', '12:00',
       '1:30', '16:45', '9:25', '4:40', '11:25', '14:25', '14:35', '9:05',
       '1:35', '10:05', '11:10', '7:45', '0:55', '23:35', '0:40', '18:50',
       '7:55', '18:15', '13:15', '9:45', '6:50', '15:15', '7:40', '2:20',
       '18:30', '19:45', '19:50', '1:00', '0:05', '16:20', '0:30', '0:50',
       '13:45', '9:15', '7:10', '8:55', '4:45', '5:35', '9:00', '10:45',
       '22:10', '22:40', '5:25', '2:15', '20:20', '6:35', '1:20', '19:10',
       '10:50', '23:25', '12:15', '10:40', '23:15', '22:55', '9:10',
       '18:10', '0:25', '20:45', '21:05', '23:00', '0:45', '4:35',
       '18:00', '12:30', '2:10', '0:20', '23:20', '12:25', '22:35',
       '20:25', '0:35', '16:50', '9:30', '12:05', '10:35', '15:05',
       '18:40', '9:35', '8:50', '16:40', '19:40', '16:10', '21:20',
       '7:25', '20:40', '13:00', '11:20', '0:10', '8:10', '16:1

Duration

In [24]:
flights.Duration

0        2h 50m
1        7h 25m
2           19h
3        5h 25m
4        4h 45m
          ...  
10678    2h 30m
10679    2h 35m
10680        3h
10681    2h 40m
10682    8h 20m
Name: Duration, Length: 10683, dtype: str

In [25]:
(
	flights
	.Duration
	.loc[lambda ser: ~ser.str.contains("m")]
	.unique()
)

<ArrowStringArray>
['19h', '23h', '22h', '12h',  '3h',  '5h', '10h', '18h', '24h', '15h', '16h',
  '8h', '14h', '20h', '13h', '11h',  '9h', '27h', '26h',  '4h',  '7h', '30h',
 '21h', '28h', '47h',  '6h', '25h', '38h', '34h']
Length: 29, dtype: str

In [26]:
(
	flights
	.Duration
	.loc[lambda ser: ~ser.str.contains("h")]
	# .unique()
)

6474    5m
Name: Duration, dtype: str

In [27]:
flights.iloc[[6474]]

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
6474,Air India,6/3/2019,Mumbai,Hyderabad,BOM → GOI → PNQ → HYD,16:50,16:55,5m,2 stops,No info,17327


  - The observation indexed 6474 has duration of 5 minutes. This is clearly wrong. Will delete this observation

In [28]:
(
	flights
	.Duration
	.drop(index=6474)
	.str.split(" ", expand=True)
	.set_axis(["hour", "minute"], axis=1)
	.assign(
		hour=lambda df_: (
			df_
			.hour
			.str.replace("h", "")
			.astype(int)
			.mul(60)
		),
		minute=lambda df_: (
			df_
			.minute
			.str.replace("m", "")
			.fillna("0")
			.astype(int)
		)
	)
	.sum(axis=1)
)

0         170
1         445
2        1140
3         325
4         285
         ... 
10678     150
10679     155
10680     180
10681     160
10682     500
Length: 10682, dtype: int64

In [29]:
(
	flights
	.Duration
	.drop(index=6474)
	.str.split(" ", expand=True)
	.set_axis(["hour", "minute"], axis=1)
	.assign(
		hour=lambda df_: (
			df_
			.hour
			.str.replace("h", "")
			.astype(int)
			.mul(60)
		),
		minute=lambda df_: (
			df_
			.minute
			.str.replace("m", "")
			.fillna("0")
			.astype(int)
		)
	)
	.sum(axis=1)
	.rename("duration_minutes")
	.to_frame()
	.join(flights.Duration.drop(index=6474))
)

,duration_minutes,Duration
0,170,2h 50m
1,445,7h 25m
2,1140,19h
3,325,5h 25m
4,285,4h 45m
...,...,...
10678,150,2h 30m
10679,155,2h 35m
10680,180,3h
10681,160,2h 40m


Total_Stops

In [30]:
flights.Total_Stops.unique()

<ArrowStringArray>
['non-stop', '2 stops', '1 stop', '3 stops', nan, '4 stops']
Length: 6, dtype: str

In [31]:
(
	flights
	.Total_Stops
	.replace("non-stop", "0")
	.str.replace(" stops?", "", regex=True) # " stop" " stops"
	.pipe(lambda ser: pd.to_numeric(ser))
)

0        0.0
1        2.0
2        2.0
3        1.0
4        1.0
        ... 
10678    0.0
10679    0.0
10680    0.0
10681    0.0
10682    2.0
Name: Total_Stops, Length: 10683, dtype: float64

additional_info

In [32]:
flights.Additional_Info.unique()

<ArrowStringArray>
[                     'No info',  'In-flight meal not included',
 'No check-in baggage included',              '1 Short layover',
                      'No Info',               '1 Long layover',
              'Change airports',               'Business class',
               'Red-eye flight',               '2 Long layover']
Length: 10, dtype: str

# 5. Cleaning Operations

In [33]:
def convert_to_minutes(ser):
	return (
		ser
		.str.split(" ", expand=True)
		.set_axis(["hour", "minute"], axis=1)
		.assign(
			hour=lambda df_: (
				df_
				.hour
				.str.replace("h", "")
				.astype(int)
				.mul(60)
			),
			minute=lambda df_: (
				df_
				.minute
				.str.replace("m", "")
				.fillna("0")
				.astype(int)
			)
		)
		.sum(axis=1)
	)

In [34]:
def clean_data(df):
	return (
		df
		.drop(index=[6474])
		.drop_duplicates()
		.assign(**{
			col: df[col].str.strip()
			for col in df.select_dtypes(include="str").columns
		})
		.rename(columns=str.lower)
		.assign(
			airline=lambda df_: (
				df_
				.airline
				.str.replace(" Premium economy", "")
				.str.replace(" Business", "")
				.str.title()
			),
			date_of_journey=lambda df_: pd.to_datetime(df_.date_of_journey, dayfirst=True),
			dep_time=lambda df_: pd.to_datetime(df_.dep_time,format="mixed",errors="coerce").dt.time,
			arrival_time=lambda df_: pd.to_datetime(df_.arrival_time,format="mixed",errors="coerce").dt.time,
			duration=lambda df_: df_.duration.pipe(convert_to_minutes),
			total_stops=lambda df_: (
				df_
				.total_stops
				.replace("non-stop", "0")
				.str.replace(" stops?", "", regex=True)
				.pipe(lambda ser: pd.to_numeric(ser))
			),
			additional_info=lambda df_: df_.additional_info.replace("No info", "No Info")
		)
		.drop(columns="route")
	)

In [35]:
flights_cleaned = clean_data(flights)
flights_cleaned

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Indigo,2019-03-24,Banglore,New Delhi,22:20:00,01:10:00,170,0.0,No Info,3897
1,Air India,2019-05-01,Kolkata,Banglore,05:50:00,13:15:00,445,2.0,No Info,7662
2,Jet Airways,2019-06-09,Delhi,Cochin,09:25:00,04:25:00,1140,2.0,No Info,13882
3,Indigo,2019-05-12,Kolkata,Banglore,18:05:00,23:30:00,325,1.0,No Info,6218
4,Indigo,2019-03-01,Banglore,New Delhi,16:50:00,21:35:00,285,1.0,No Info,13302
...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,2019-04-09,Kolkata,Banglore,19:55:00,22:25:00,150,0.0,No Info,4107
10679,Air India,2019-04-27,Kolkata,Banglore,20:45:00,23:20:00,155,0.0,No Info,4145
10680,Jet Airways,2019-04-27,Banglore,Delhi,08:20:00,11:20:00,180,0.0,No Info,7229
10681,Vistara,2019-03-01,Banglore,New Delhi,11:30:00,14:10:00,160,0.0,No Info,12648


# 6. Split the data

In [43]:
flights_final = flights_cleaned.sample(1000)

In [44]:
X = flights_final.drop(columns="price")
y = flights_final.price.copy()

In [45]:
X_, X_test, y_, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_, y_, test_size=0.2, random_state=42)

In [46]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(640, 9) (640,)
(160, 9) (160,)
(200, 9) (200,)


# 7. Export the Subsets

In [47]:
def export_data(X, y, name):
	file_name = f"{name}.csv"
	file_path = os.path.join(PROJECT_DIR, DATA_DIR, file_name)

	X.join(y).to_csv(file_path, index=False)

	return pd.read_csv(file_path).head()

In [48]:
export_data(X_train, y_train, "train")

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Indigo,2019-04-27,Delhi,Cochin,16:00:00,21:00:00,300,1.0,No Info,6937
1,Vistara,2019-04-01,Kolkata,Banglore,07:10:00,23:25:00,975,1.0,No Info,7927
2,Jet Airways,2019-05-27,Delhi,Cochin,14:00:00,19:00:00,300,1.0,In-flight meal not included,12898
3,Jet Airways,2019-06-24,Delhi,Cochin,20:55:00,19:00:00,1325,1.0,In-flight meal not included,10262
4,Jet Airways,2019-05-18,Delhi,Cochin,14:00:00,19:00:00,300,1.0,In-flight meal not included,12373


In [49]:
export_data(X_val, y_val, "val")

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Jet Airways,2019-05-27,Delhi,Cochin,14:00:00,04:25:00,865,1.0,In-flight meal not included,12898
1,Air Asia,2019-06-27,Banglore,Delhi,23:55:00,02:45:00,170,0.0,No Info,3383
2,Jet Airways,2019-05-18,Delhi,Cochin,18:15:00,04:25:00,610,1.0,In-flight meal not included,12373
3,Indigo,2019-05-27,Delhi,Cochin,06:40:00,21:00:00,860,1.0,No Info,6526
4,Jet Airways,2019-06-09,Delhi,Cochin,05:30:00,04:25:00,1375,2.0,No Info,13292


In [50]:
export_data(X_test, y_test, "test")

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Indigo,2019-05-06,Kolkata,Banglore,06:20:00,10:20:00,240,1.0,No Info,4226
1,Vistara,2019-05-18,Kolkata,Banglore,07:10:00,18:50:00,700,1.0,No Info,9345
2,Indigo,2019-06-03,Mumbai,Hyderabad,19:05:00,20:35:00,90,0.0,No Info,2754
3,Multiple Carriers,2019-03-21,Delhi,Cochin,17:00:00,01:35:00,515,1.0,No Info,7363
4,Air India,2019-05-15,Kolkata,Banglore,09:25:00,21:50:00,745,1.0,No Info,8366
